# Dental Detection RAG Assistant

Adds a Retrieval-Augmented Generation (RAG) layer on top of the YOLOv8 tooth-condition
detector (`Cavity`, `Fillings`, `Impacted Tooth`, `Implant`) from `YOLO.ipynb` / `teeth.ipynb`.

**What this notebook does**
1. Builds a small dental knowledge base (chunked text) that you can swap for real textbook PDFs.
2. Embeds the chunks and stores them in a vector index (FAISS).
3. Wraps retrieval + an LLM in a `RetrievalQA` chain with a dental-assistant system prompt.
4. Exposes `explain_detection(class_name)` so that whatever YOLO detects can be automatically
   explained using the retrieved context (with sources cited).
5. Also supports free-form Q&A for a dentist/patient to ask questions.

**Fully local setup — no API key or billing required**: embeddings run locally via
`sentence-transformers` (`all-MiniLM-L6-v2`), and the answering LLM runs locally via
[Ollama](https://ollama.com) (`llama3.2:1b` by default — a small model that runs on modest
hardware; swap to a bigger Ollama model if your machine has more RAM). You need Ollama
installed and the model pulled once (`ollama pull llama3.2:1b`) — see the install cell below —
but nothing to sign up for or pay for.

**Knowledge base sources**: the reference entries below are written in our own words but are
each grounded in and cited to a real, freely accessible dental source (NCBI Bookshelf /
StatPearls for caries and implants, peer-reviewed restorative-materials studies for fillings,
clinical reference glossaries for impacted teeth) — see the `source_url` in each entry. This
avoids reproducing copyrighted textbook text wholesale while still grounding answers in real
literature. If your instructor wants a specific textbook used, drop its PDF into the folder in
the loader cell below and it will be chunked and added to the same index alongside these.

## 0. One-time local setup (do this before running any cells)

1. Install [Ollama](https://ollama.com/download) for your OS (Windows/Mac/Linux) — it's a free
   app that runs LLMs locally, no account or API key needed.
2. Open a terminal and run:
   ```
   ollama pull llama3.2:1b
   ```
   This downloads the model once (a few GB). Leave the Ollama app running in the background.
3. Come back here and run the cells below in order.

In [28]:
# 1. Install Python dependencies (run once)
%pip install -q langchain langchain-community langchain-ollama \
    sentence-transformers faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


In [29]:
# 2. Imports — no API keys needed, everything below runs locally
import os
import glob

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

print("Imports ready. ✅ (no API key required — embeddings and LLM both run locally)")

Imports ready. ✅ (no API key required — embeddings and LLM both run locally)


## 3. Knowledge base

`DENTAL_DOCS` below is a small demo corpus: one entry per class your YOLO model detects, written
as original educational summaries (definition, causes, radiographic appearance, treatment).
This is enough to make the RAG pipeline work end-to-end. Swap it out for real textbook content
using the PDF loader cell right after it.

In [30]:
# 3a. Dental knowledge base — summaries grounded in real, open-access dental literature.
# Each entry is written in our own words (not copy-pasted) but is based on and cited to a real
# source, so the "sources" your RAG chain prints back are genuine references, not placeholders.
# Swap/add PDF textbook chunks below via the loader cell if your instructor requires specific texts.
DENTAL_DOCS = [
    {
        "class": "Cavity",
        "title": "Dental Caries — StatPearls (NCBI Bookshelf, NBK551699)",
        "source_url": "https://www.ncbi.nlm.nih.gov/books/NBK551699/",
        "text": (
            "Dental caries (a cavity) is a chronic, bacteria-driven disease in which "
            "tooth-adherent cariogenic bacteria, chiefly Streptococcus mutans, metabolize "
            "dietary sugars into acid, which over time demineralizes and dissolves enamel and "
            "dentin. The term describes both the disease process and the resulting lesion. "
            "Untreated, the lesion progresses from an early enamel change into a true cavity "
            "that can extend into dentin and eventually the pulp, causing sensitivity and pain. "
            "On a dental radiograph, carious lesions typically show up as a darker "
            "(radiolucent) area within the crown or root, because demineralized tissue "
            "attenuates X-rays less than healthy mineralized tooth structure. Management "
            "requires an interprofessional approach: early lesions can sometimes be arrested "
            "or remineralized, while established cavities need the decayed tissue removed and "
            "the tooth restored; deep decay reaching the pulp may require root canal therapy."
        ),
    },
    {
        "class": "Fillings",
        "title": "Radiopacity of Dental Restorative Materials (Acta Odontologica Scandinavica; Operative Dentistry)",
        "source_url": "https://medicaljournalssweden.se/actaodontologica/article/view/38897",
        "text": (
            "A dental filling restores a tooth after decayed or damaged tissue has been "
            "removed, using materials such as dental amalgam, composite resin, or glass "
            "ionomer cement. Radiographically, these materials differ a lot in how "
            "radiopaque (bright/white) they appear. Amalgam is highly radiopaque and shows up "
            "as a strongly bright area on an X-ray. Composite resins are not intrinsically "
            "radiopaque; manufacturers add high-atomic-number fillers such as barium, "
            "strontium, zinc, zirconium, or ytterbium so the material can be distinguished from "
            "tooth structure and recurrent decay can be detected under the restoration. Studies "
            "measuring restorative materials against an aluminum step-wedge reference have "
            "found wide variation: some composites are more radiopaque than enamel, others "
            "less radiopaque than dentin, and glass ionomer and zinc-phosphate-based materials "
            "generally sit between resin composites and amalgam. International standards "
            "(ISO 4049) set minimum radiopacity requirements for resin-based filling materials "
            "specifically so recurrent caries at a filling's margin remains detectable on X-ray."
        ),
    },
    {
        "class": "Impacted Tooth",
        "title": "Impacted Tooth — clinical reference summaries (Overjet glossary; Encyclopedia.com/Gale)",
        "source_url": "https://www.overjet.com/glossary/impacted-tooth",
        "text": (
            "An impacted tooth is one that fails to fully erupt into its normal position in the "
            "arch because of lack of space, misalignment, or a physical obstruction such as an "
            "adjacent tooth or bone. Third molars (wisdom teeth) are impacted most often, "
            "followed by maxillary canines; impactions are commonly classified by the "
            "direction the tooth is angled — mesioangular, distoangular, vertical, or "
            "horizontal. On radiographs an impacted tooth appears lodged within bone, often "
            "angled against or beneath a neighboring tooth's roots, sometimes with a "
            "radiolucent follicular space around the crown. Impacted teeth can be asymptomatic "
            "or cause pain, gum swelling and tenderness, infection, cyst formation, or "
            "resorption of adjacent tooth roots; panoramic radiographs are the standard way to "
            "assess position and decide whether extraction (common for wisdom teeth) or "
            "orthodontic guidance (sometimes used for impacted canines) is appropriate."
        ),
    },
    {
        "class": "Implant",
        "title": "Dental Implants — StatPearls (NCBI Bookshelf, NBK470448)",
        "source_url": "https://www.ncbi.nlm.nih.gov/books/NBK470448/",
        "text": (
            "A dental implant is an alloplastic (e.g., titanium) structure placed into the "
            "jawbone beneath the mucosa and/or periosteum to support a fixed or removable "
            "prosthetic tooth, and is one of the main treatments for replacing missing teeth. "
            "Reported advantages over a conventional fixed partial denture include a high "
            "long-term success rate (above 97% at 10 years), decreased risk of caries and "
            "endodontic problems in adjacent teeth (since they aren't used as bridge "
            "abutments), and better preservation of bone at the edentulous site. Implants work "
            "through osseointegration — a direct structural bond between living bone and the "
            "implant surface — which needs to stabilize (typically over a period of months) "
            "before the final crown or prosthesis is attached. On a radiograph, implants are "
            "strongly radiopaque and have a distinct regular, screw-like or cylindrical shape "
            "that makes them easy to tell apart from natural tooth roots or metallic fillings. "
            "Long-term success depends on adequate bone volume, good oral hygiene, and avoiding "
            "peri-implantitis (inflammation and bone loss around the implant)."
        ),
    },
]

print(f"Loaded {len(DENTAL_DOCS)} reference entries grounded in cited dental literature.")
for d in DENTAL_DOCS:
    print(f" - {d['class']}: {d['title']}")

Loaded 4 reference entries grounded in cited dental literature.
 - Cavity: Dental Caries — StatPearls (NCBI Bookshelf, NBK551699)
 - Fillings: Radiopacity of Dental Restorative Materials (Acta Odontologica Scandinavica; Operative Dentistry)
 - Impacted Tooth: Impacted Tooth — clinical reference summaries (Overjet glossary; Encyclopedia.com/Gale)
 - Implant: Dental Implants — StatPearls (NCBI Bookshelf, NBK470448)


In [31]:
# 3b. OPTIONAL: load real textbook/course-material PDFs instead of (or in addition to) the demo docs.
# Put your PDF files in this folder, then run this cell.
PDF_FOLDER = r"C:\Users\landa\Desktop\dental_textbooks"  # <-- change to your PDFs folder

pdf_documents = []
if os.path.isdir(PDF_FOLDER):
    for pdf_path in glob.glob(os.path.join(PDF_FOLDER, "*.pdf")):
        loader = PyPDFLoader(pdf_path)
        pdf_documents.extend(loader.load())
    print(f"Loaded {len(pdf_documents)} pages from PDFs in {PDF_FOLDER}")
else:
    print("No PDF folder found — continuing with the demo knowledge base only.")

No PDF folder found — continuing with the demo knowledge base only.


In [32]:
# 4. Chunk everything into Documents with metadata
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_documents = []

# Reference corpus -> Documents
for entry in DENTAL_DOCS:
    for chunk in splitter.split_text(entry["text"]):
        all_documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "class": entry["class"],
                    "source": entry["title"],
                    "source_url": entry.get("source_url", ""),
                },
            )
        )

# Any loaded PDFs -> Documents (already Document objects, just re-split)
for doc in pdf_documents:
    for chunk in splitter.split_text(doc.page_content):
        meta = dict(doc.metadata)
        meta.setdefault("class", "Unknown")
        all_documents.append(Document(page_content=chunk, metadata=meta))

print(f"Total chunks: {len(all_documents)}")

Total chunks: 12


In [33]:
# 5. Embeddings + vector store (local, free — downloads a small model the first time only)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(all_documents, embeddings)

# Save/load so you don't have to re-embed every run:
vectorstore.save_local("dental_faiss_index")
# vectorstore = FAISS.load_local("dental_faiss_index", embeddings, allow_dangerous_deserialization=True)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready. ✅")

Vector store ready. ✅


## 6. Prompt template and QA chain

Same shape as a standard LangChain `RetrievalQA` setup: a system prompt defining the assistant's
role, a template that injects retrieved context, and a chain that ties the retriever + LLM
together.

In [34]:
# 6a. System prompt + RAG prompt template (dental-assistant version)
SYSTEM_PROMPT = """\
You are a Dental Assistant AI supporting a tooth-condition detection system that flags
Cavity, Fillings, Impacted Tooth, and Implant findings on dental X-rays.

Your personality: friendly, clear, and professional — like a knowledgeable dental
assistant explaining a finding to a patient or a junior colleague.

Your responsibilities:
1. EXPLAIN DETECTIONS → Given a detected class, explain what it is, likely causes,
   how it typically appears on an X-ray, and common treatment options.
2. ANSWER QUESTIONS → Answer general dental questions using the retrieved context
   whenever it's relevant.
3. STAY GROUNDED → Base clinical claims on the provided context. If the context doesn't
   cover something, say so rather than guessing.
4. NOT A DIAGNOSIS → Always note this is educational information, not a diagnosis or
   substitute for an in-person dental professional.
"""

RAG_TEMPLATE = """\
{system_prompt}

--- Relevant Context from Dental Knowledge Base ---
{context}
----------------------------------------------------

Question: {question}

Answer directly (no greeting, cite the source titles you used at the end):"""

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    partial_variables={"system_prompt": SYSTEM_PROMPT},
    template=RAG_TEMPLATE,
)

print("Prompt template ready. ✅")

Prompt template ready. ✅


In [35]:
# 6b. LLM + RetrievalQA chain (local Ollama model — no API key needed)
# Make sure Ollama is running and you've pulled the model: `ollama pull llama3.2:1b`
# llama3.2:1b is small and runs on modest RAM. If your machine has more RAM (8GB+ free),
# you can use a larger, higher-quality model instead, e.g. model="llama3.2" (3B).
llm = ChatOllama(model="llama3.2:1b", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("QA chain ready. ✅")

QA chain ready. ✅


## 7. Auto-explain YOLO detections

`explain_detection` runs one class through the RAG chain. `explain_yolo_results` takes the
output of `model.predict(...)` (from `YOLO.ipynb`) and explains every unique class detected in
an image.

In [36]:
# 7a. Explain a single detected class
def explain_detection(class_name: str):
    query = (
        f"A dental X-ray detection model flagged '{class_name}'. "
        f"Explain what this finding is, its likely causes, how it typically looks on an "
        f"X-ray, and common treatment options."
    )
    result = qa_chain.invoke({"query": query})
    print(f"=== {class_name} ===")
    print(result["result"])
    seen = {}
    for doc in result["source_documents"]:
        seen[doc.metadata.get("source", "unknown")] = doc.metadata.get("source_url", "")
    print("\nSources:")
    for title, url in seen.items():
        print(f"  - {title}" + (f" ({url})" if url else ""))
    print()
    return result

# Quick test
_ = explain_detection("Impacted Tooth")

=== Impacted Tooth ===
An impacted tooth is a tooth that fails to fully erupt into its normal position in the arch due to lack of space, misalignment, or a physical obstruction. This can occur when there is insufficient room for the tooth to erupt, resulting in it being lodged within the bone.

The direction of the tooth's angulation is crucial in determining the type of impacted tooth. Impacted teeth can be classified into four main categories:

- Mesioangular: The tooth is angled towards the midline of the jaw.
- Distoangular: The tooth is angled towards the outer aspect of the jaw.
- Vertical: The tooth is angled vertically, often with the crown positioned below the level of the surrounding bone.
- Horizontal: The tooth is angled horizontally, often with the crown positioned above the level of the surrounding bone.

On a dental radiograph, an impacted tooth typically appears lodged within the bone, often angled against or beneath a neighboring tooth's roots. There may be a radioluce

In [37]:
# 7b. Explain every class found in a YOLO prediction
# Example usage once you have a trained model from YOLO.ipynb:
#
# from ultralytics import YOLO
# model = YOLO("path/to/best.pt")
# results = model.predict(source="path/to/xray.jpg")
#
# CLASS_NAMES = ['Implant', 'Fillings', 'Impacted', 'Cavity']  # match your data.yaml order

def explain_yolo_results(results, class_names):
    detected_classes = set()
    for r in results:
        for box in r.boxes:
            class_id = int(box.cls[0])
            detected_classes.add(class_names[class_id])

    if not detected_classes:
        print("No detections to explain.")
        return {}

    return {cls: explain_detection(cls) for cls in detected_classes}

print("explain_yolo_results() ready — call it with your model's predict() output.")

explain_yolo_results() ready — call it with your model's predict() output.


## 8. Free-form Q&A

For a dentist or patient to ask their own questions, grounded in the same knowledge base.

In [38]:
def ask(question: str):
    result = qa_chain.invoke({"query": question})
    print(result["result"])
    seen = {}
    for doc in result["source_documents"]:
        seen[doc.metadata.get("source", "unknown")] = doc.metadata.get("source_url", "")
    print("\nSources:")
    for title, url in seen.items():
        print(f"  - {title}" + (f" ({url})" if url else ""))
    return result

# Example:
_ = ask("What's the difference between a cavity and an impacted tooth on an X-ray?")

I'd be happy to explain the difference between a cavity and an impacted tooth on an X-ray.

A cavity is a demineralized area within the tooth's crown or root, typically caused by the decay of tooth structure due to acid erosion or other factors. On an X-ray, carious lesions usually appear as a darker (radiolucent) area within the crown or root, as demineralized tissue attenuates X-rays less than healthy mineralized tooth structure.

An impacted tooth, on the other hand, is a tooth that fails to fully erupt into its normal position in the arch due to lack of space, misalignment, or a physical obstruction. Impacted teeth can be asymptomatic or cause pain, gum swelling and tenderness, infection, cyst formation, or resorption of adjacent tooth roots.

The key difference between a cavity and an impacted tooth is the underlying cause and the typical appearance on an X-ray. Caries are usually localized to the tooth structure, while impacted teeth are often more complex and may involve surroun

## Notes for the write-up

- **Retrieval**: FAISS similarity search over local `sentence-transformers/all-MiniLM-L6-v2`
  embeddings, top-`k=4` chunks. No API key or billing — runs entirely on your machine.
- **Generation**: `RetrievalQA` (LangChain) using a local `llama3.2:1b` model via Ollama, with a
  custom prompt that grounds answers in retrieved context and asks the model to cite sources.
- **Integration point**: `explain_yolo_results()` connects directly to the `ultralytics` model's
  `predict()` output from `YOLO.ipynb`, so each bounding-box class detected by YOLO automatically
  triggers a grounded explanation.
- **Knowledge base**: `DENTAL_DOCS` entries are grounded in and cited to real dental references
  (NCBI StatPearls, peer-reviewed restorative-materials studies, clinical glossaries) rather than
  invented text, so retrieved answers cite genuine sources (see `source_url` per entry). If your
  course requires a specific assigned textbook, drop its PDF into `PDF_FOLDER` and re-run the
  chunking + embedding cells to fold it into the same index.
- **Want to use OpenAI/Anthropic instead?** Swap `HuggingFaceEmbeddings` for `OpenAIEmbeddings`
  and `ChatOllama` for `ChatOpenAI`/`ChatAnthropic`, then set the matching API key — but the
  local setup above needs none of that.